# U-Net Pipeline Applied

## Set Up

In [1]:
import os

# Assuming your original notebook is running on GPU 0, point this to GPU 1
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Verify the setting (optional)
print(f"Targeting GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")

Targeting GPU: 0


In [2]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

/hpc/jgeo610/Virtual_ENV/unet_env/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


2.5.1+cu118
True
Tesla V100-PCIE-32GB


In [3]:
print(torch.cuda.current_device())
print(torch.cuda.get_device_name())

0
Tesla V100-PCIE-32GB


In [4]:
import sys
print(sys.executable)

/hpc/jgeo610/Virtual_ENV/unet_env/bin/python


In [5]:
import sys
sys.path.append('.')

import logging
import torch
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [6]:
import train
from unet import UNet

train.dir_img = Path('./data/imgs/')
train.dir_mask = Path('./data/masks/')
train.dir_checkpoint = Path('./checkpoints/')

# 'fat' keeps only EAT (the water notebook, unet_applied.ipynb, is untouched by
# this) -- see VolumeMRIDataset.phase / CLAUDE.md.
PHASE = 'fat'
N_CLASSES = train.PHASE_N_CLASSES[PHASE]   # 2 for fat (background + EAT)

# train_model prefixes non-water run names with '{phase}_' (water itself stays
# unprefixed, for backward compatibility with checkpoints that predate `phase`
# existing at all). Cells below that build a checkpoint path THEMSELVES, rather
# than through train_model, replicate that same rule via this helper so their
# bookkeeping (already_done, loading CSVs, ...) lands on the same directory
# train_model actually used.
def run_dir_for(run_name):
    return train.dir_checkpoint / (run_name if PHASE == 'water' else f'{PHASE}_{run_name}')

<!-- ## Training U-Net Model -->

In [ ]:
SEEDS = [0, 1, 2]   # vary these (with split_seed fixed) to measure run-to-run variance.
                     # The baseline and augmented training cells below each loop over
                     # SEEDS independently, seeding and building a FRESH model per seed.

# seed BEFORE constructing the model - UNet(...) draws its initial weights from
# the torch RNG, so seeding afterwards leaves the init random and two otherwise
# identical runs will still differ. This one model is just for the sanity-check
# print below; the training cells build their own per seed.
train.set_seed(SEEDS[0])

model = UNet(n_channels=1, n_classes=N_CLASSES, bilinear=False)
model = model.to(memory_format=torch.channels_last)
model = model.to(device=device)

INFO: Seeded RNGs with 1


In [ ]:
# import numpy as np
# from pathlib import Path

# cache_dir = train.dir_img.parent / 'preprocessed_cache'
# patient_ids = [p.stem for p in sorted(train.dir_img.glob('*.dcm'))]

# corrupted = []
# for patient_id in patient_ids:
#     img_path = cache_dir / f'{patient_id}_img.npy'
#     mask_path = cache_dir / f'{patient_id}_mask.npy'
#     for p in [img_path, mask_path]:
#         if p.exists():
#             try:
#                 np.load(p, mmap_mode='r').shape
#             except Exception as e:
#                 print(f'CORRUPTED, deleting: {p} ({e})')
#                 p.unlink()
#                 corrupted.append(p)

# print(f'\nRemoved {len(corrupted)} corrupted file(s).')

In [ ]:
print(model.n_classes)

In [ ]:
import numpy as np
from utils.data_loading import BasicDataset, VolumeMRIDataset

dataset = VolumeMRIDataset(train.dir_img, train.dir_mask, scale=1.0, phase=PHASE)

In [ ]:
# import matplotlib.pyplot as plt
# sample = dataset[50]
# mask = sample['mask'].numpy()
# plt.figure()
# plt.imshow(mask)
# plt.show()


In [ ]:
# import matplotlib.pyplot as plt
# sample = dataset[50]  # or any index where fat is visible
# mask = sample['mask']
# for c in dataset.mask_values:
#     plt.figure()
#     plt.imshow(mask == int(c))
#     plt.title(f'Class {int(c)}')
#     plt.show()

In [ ]:
# sample = dataset[0]
# img = sample['image']
# print('shape:', img.shape)
# print('dtype:', img.dtype)
# print('min:', img.min().item(), 'max:', img.max().item())

In [ ]:
# for i in [0, 100, 500, 1000, 2000]:
#     img = dataset[i]['image']
#     print(i, img.min().item(), img.max().item(), torch.isnan(img).any().item(), torch.isinf(img).any().item())

## Visualise a slice

In [ ]:
# import matplotlib.pyplot as plt

# # 1. Fetch a specific slice index from your dataset instance
# # (e.g., dataset[0] for the first slice, or dataset[50] for slice 50)
# sample_idx = 50  
# sample = dataset[sample_idx]

# # 2. Extract image and mask tensors
# # image shape is (1, H, W) -> squeeze to (H, W) for plotting
# image_2d = sample['image'].squeeze(0).numpy()  
# mask_2d = sample['mask'].numpy()               # shape: (H, W)

# # 3. Plot image and mask side-by-side
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# # Plot normalized MRI slice
# axes[0].imshow(image_2d, cmap='gray')
# axes[0].set_title(f'MRI Image (Index {sample_idx})')
# axes[0].axis('off')

# # Plot segmentation mask overlay
# axes[1].imshow(mask_2d, cmap='tab20')  # 'tab20' gives clear distinct colors for integer classes
# axes[1].set_title(f'Mask (Index {sample_idx})')
# axes[1].axis('off')

# plt.tight_layout()
# plt.show()

In [ ]:
# patient_id = sample['patient_id']
# print(f"Loaded Patient: {patient_id}")

## Visualise specific patient and slice

In [ ]:
# import matplotlib.pyplot as plt

# # 1. Target patient and slice
# target_patient = "CADRE_1645"
# target_slice = 50

# # 2. Find the exact dataset index
# try:
#     sample_idx = next(
#         idx for idx, (p_id, s_idx) in enumerate(dataset.index)
#         if p_id == target_patient and s_idx == target_slice
#     )
# except StopIteration:
#     raise ValueError(f"Slice {target_slice} for patient '{target_patient}' was not found in dataset.index.")

# # 3. Load the sample
# # print("hello")
# sample = dataset[sample_idx]
# # print("bye")
# # print("hello1")
# image_2d = sample['image'].squeeze(0).numpy()
# mask_2d = sample['mask'].numpy()
# # print("bye1")

# # 4. Display the slice and mask side-by-side
# fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# axes[0].imshow(image_2d, cmap='gray')
# axes[0].set_title(f"Patient: {target_patient} | Slice {target_slice}")
# axes[0].axis('off')

# axes[1].imshow(mask_2d, cmap='tab20')
# axes[1].set_title("Segmentation Mask")
# axes[1].axis('off')

# plt.tight_layout()
# plt.show()

In [ ]:
# import numpy as np
# from utils.data_loading import BasicDataset, VolumeMRIDataset

# dataset = VolumeMRIDataset(train.dir_img, train.dir_mask, scale=1.0)

# for patient_id in dataset.mask_file_for:
#     img_path, mask_path = dataset._disk_cache_paths(patient_id)
#     for p in [img_path, mask_path]:
#         if p.exists():
#             try:
#                 np.load(p, mmap_mode='r').shape   # cheap check, doesn't load full array into memory
#             except Exception as e:
#                 print(f'CORRUPTED, deleting: {p} ({e})')
#                 p.unlink()

#     print('Done checking.')

In [ ]:
#from utils.data_loading import BasicDataset, VolumeMRIDataset

#dataset = VolumeMRIDataset(train.dir_img, train.dir_mask, scale=1.0)
# for patient_id in dataset.mask_file_for:
#     dataset._get_volume(patient_id)   # forces disk-cache creation for every patient, up front
# print('All patients cached to disk.')

In [ ]:
# import torch

# print("PyTorch:", torch.__version__)
# print("CUDA available:", torch.cuda.is_available())
# print("CUDA version:", torch.version.cuda)
# print("cuDNN version:", torch.backends.cudnn.version())
# print("cuDNN enabled:", torch.backends.cudnn.enabled)

# print(torch.cuda.get_device_name(0))

In [ ]:
# import torch

# print(torch.__version__)
# print(torch.backends.cudnn.version())

In [ ]:
# import torch
# import torch.nn as nn

# x = torch.randn(8, 1, 432, 432).cuda(0)

# conv = nn.Conv2d(1, 64, kernel_size=3, padding=1).cuda(0)

# y = conv(x)

# print(y.shape)

In [ ]:
# import torch

# print("PyTorch version:", torch.__version__)
# print("CUDA runtime:", torch.version.cuda)
# print("cuDNN:", torch.backends.cudnn.version())
# print("cuDNN enabled:", torch.backends.cudnn.enabled)

# print("GPU:", torch.cuda.get_device_name(0))
# print("Capability:", torch.cuda.get_device_capability(0))

In [ ]:
#train.train_model(model=model, device=device, epochs=1, batch_size=15, learning_rate=1e-5, val_percent=0.1, img_scale=1.0, amp=False)

In [ ]:
print(os.environ.get('WANDB_MODE'))

In [ ]:
# import threading
# import time
# import debugpy  # Handles the VS Code debugger pause
# import pynvml

# # Allow VS Code to attach to this notebook session on port 5678
# debugpy.configure(python='python')
# debugpy.listen(('localhost', 5678))

# def monitor_gpu_and_pause():
#     pynvml.nvmlInit()
#     # Get the first GPU (change index to 1, 2 etc. if you use a different card)
#     handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    
#     print("GPU Monitor started. Will pause debugger if GPU utilisation hits 0%.")
    
#     zero_count = 0
#     while True:
#         # Fetch current GPU utilisation percentage
#         util = pynvml.nvmlDeviceGetUtilizationRates(handle).gpu
        
#         if util == 0:
#             zero_count += 1
#         else:
#             zero_count = 0  # Reset if there is activity
            
#         # If GPU stays at 0% for ~3 consecutive checks, trigger the breakpoint
#         if zero_count >= 3:
#             print(f"\n[ALERT] GPU Utilisation is {util}%. Pausing execution...")
#             debugpy.breakpoint()  # This forces the VS Code debugger to pause
#             break
            
#         time.sleep(1)  # Check every second

# # Start the monitor in the background
# gpu_thread = threading.Thread(target=monitor_gpu_and_pause, daemon=True)
# gpu_thread.start()


In [ ]:
# import numpy as np
# import nibabel as nib
# from collections import defaultdict

# class_voxel_counts = defaultdict(int)
# total_voxels = 0

# for patient_id, mask_path in dataset.mask_file_for.items():
#     vol = nib.load(mask_path).get_fdata()
#     unique, counts = np.unique(vol, return_counts=True)
#     for val, count in zip(unique, counts):
#         class_voxel_counts[val] += int(count)
#     total_voxels += vol.size

# print(f'Total voxels: {total_voxels:,}\n')
# print(f'{"Class":<10}{"Voxel count":<18}{"Percentage":<12}')
# for cls in sorted(class_voxel_counts.keys()):
#     count = class_voxel_counts[cls]
#     pct = 100 * count / total_voxels
#     print(f'{cls:<10}{count:<18,}{pct:<12.4f}')

In [ ]:
# The corrected losses landed AFTER the original baseline_3split /
# data_aug_3split runs, so those directories hold pre-fix results. New run names
# here so both survive and stay distinguishable.
#   loss='dice_ce'         per-class batch Dice (background excluded) + CE
#   loss='dice_ce_legacy'  reproduces the pooled Dice the old runs used
#
# eval_test=False, as in loss_ablation.ipynb: comparing baseline against
# augmentation is model selection, so it runs on validation only. The test set is
# scored once, at the end, after everything is decided.
#
# Loops over SEEDS, resumable like loss_ablation.ipynb's run_arm(): a seed whose
# run_config.json already contains 'finished' is skipped and reloaded from disk
# instead of retrained. Delete checkpoints/<run_name>/ to force a genuine re-run.
import json

def baseline_run_name(seed):
    return f'baseline_dice_ce_s{seed}'

def already_done(run_name):
    cfg = run_dir_for(run_name) / 'run_config.json'
    if not cfg.exists():
        return False
    try:
        return 'finished' in json.loads(cfg.read_text())
    except json.JSONDecodeError:
        return False

baseline_results = {}   # seed -> train_model() result dict (or reloaded config)

for seed in SEEDS:
    run_name = baseline_run_name(seed)
    run_dir = run_dir_for(run_name)

    if already_done(run_name):
        cfg = json.loads((run_dir / 'run_config.json').read_text())
        print(f'{run_name} already finished (best epoch {cfg["best_epoch"]}, '
              f'macro Dice {cfg.get("best_val_macro_dice", float("nan")):.4f}). Skipping.')
        baseline_results[seed] = cfg
        continue

    # seed BEFORE constructing the model - UNet(...) draws its initial weights
    # from the torch RNG, so seeding afterwards leaves the init random and two
    # otherwise identical runs will still differ
    train.set_seed(seed)
    model = UNet(n_channels=1, n_classes=N_CLASSES, bilinear=False)
    model = model.to(memory_format=torch.channels_last).to(device=device)

    res = train.train_model(
        model=model,
        device=device,
        dataset=dataset,        # built above -- avoids re-parsing all 50 volumes
        epochs=40,
        batch_size=8,
        learning_rate=1e-5,
        img_scale=1.0,
        amp=True,
        # three-way split, by patient. split_seed must stay fixed across runs,
        # otherwise the "held-out" test patients change and stop being held out.
        val_percent=0.15,
        test_percent=0.15,
        split_seed=0,
        augment=False,          # set True to train with rotation/elastic/contrast/noise
        loss='dice_ce',         # stated explicitly: the arm belongs in the record, not in a default
        select_on='macro_dice', # best.pth picked on per-patient volume Dice, not slice-wise
        lr_schedule='poly',     # nnU-Net decay. The old plateau scheduler stepped ~5x/epoch
                                 # and collapsed the LR 100x by epoch 5 -- see CLAUDE.md
        eval_test=False,        # test stays sealed
        run_name=run_name,
        phase=PHASE,             # picks the slice half + mask relabeling
        seed=seed,               # model init, batch order, augmentation
    )
    baseline_results[seed] = res

    print(f'{run_name}: best val Dice (slice) {res["best_val_dice"]:.4f}  (epoch {res["best_epoch"]})')
    print(f'{run_name}: best val Dice (macro) {res["best_val_macro_dice"]:.4f}   <- this is what selected best.pth')
    print(f'checkpoints           : {res["checkpoint_dir"]}')

# Downstream cells in this notebook were written against a single BASELINE_RUN /
# results -- point them at the last seed trained above. Change this if you want
# to inspect a different seed's run.
SEED = SEEDS[-1]
BASELINE_RUN = baseline_run_name(SEED)
results = baseline_results[SEED]

<!-- ## Evaluation Final Valdiation Score -->

In [ ]:
# train_model already scored validation per patient in 3D (Dice + HD95 in mm)
# and wrote the CSVs next to the checkpoints. This is the reportable table.
#
# Only 'val' is present: eval_test=False above, so the test set was not touched.
# It gets scored once, in the gated cell at the bottom of this notebook.
from utils import metrics

print(metrics.format_summary(results['per_patient_metrics']['val']['summary']))
print('\nCSVs written to:', results['checkpoint_dir'])

In [ ]:
print('val patients :', results['val_patients'])
print('test patients:', results['test_patients'])
print(f"sizes -> train {results['n_train']} / val {results['n_val']} / test {results['n_test']} slices")

In [ ]:
# Rebuild the exact split train_model used, for the inspection cells below.
from torch.utils.data import Subset, DataLoader
from utils.data_loading import VolumeMRIDataset

# these must match what you passed to train_model above
img_scale    = 1.0
val_percent  = 0.15
test_percent = 0.15
split_seed   = 0
batch_size   = 8

dataset = VolumeMRIDataset(train.dir_img, train.dir_mask, scale=img_scale, phase=PHASE)

# same function train_model calls, so these are exactly the same patients
train_idx, val_idx, test_idx = train.split_patients(
    dataset, val_percent=val_percent, test_percent=test_percent, seed=split_seed
)

val_set  = Subset(dataset, val_idx)
test_set = Subset(dataset, test_idx)

# drop_last=False: dropping the final partial batch would silently discard
# validation slices and bias the score
val_loader = DataLoader(val_set, shuffle=False, drop_last=False,
                        batch_size=batch_size, num_workers=0, pin_memory=True)

print(f'{len(dataset.mask_file_for)} patients, {len(dataset.index)} slices')
print(f'mask values: {dataset.mask_values}')
print(f'{len(val_set)} validation slices / {len(test_set)} test slices')

In [ ]:
train_set = Subset(dataset, train_idx)
print(f'{len(train_set)} training slices')

In [ ]:
# A FRESH model per seed, seeded exactly like the baseline.
#
# This cell used to pass `model` -- the same object the baseline cell had already
# trained for 40 epochs -- so this run was really 40 baseline epochs followed by
# 40 augmented ones, not an independent arm. It also fell back to the default
# seed instead of SEED. Both are fixed below, so augmentation is the only
# difference between each of these runs and its baseline counterpart at the same
# seed.
#
# Loops over SEEDS the same resumable way the baseline cell above does --
# `already_done` and `json` are defined there and reused here rather than
# repeated, but this stays its own cell/loop so it can be run and watched
# independently of the baseline.
def aug_run_name(seed):
    return f'data_aug_dice_ce_s{seed}'

aug_results = {}   # seed -> train_model() result dict (or reloaded config)

for seed in SEEDS:
    run_name = aug_run_name(seed)
    run_dir = run_dir_for(run_name)

    if already_done(run_name):
        cfg = json.loads((run_dir / 'run_config.json').read_text())
        print(f'{run_name} already finished (best epoch {cfg["best_epoch"]}, '
              f'macro Dice {cfg.get("best_val_macro_dice", float("nan")):.4f}). Skipping.')
        aug_results[seed] = cfg
        continue

    train.set_seed(seed)        # BEFORE UNet(...) -- it draws its init from the torch RNG
    model_aug = UNet(n_channels=1, n_classes=N_CLASSES, bilinear=False)
    model_aug = model_aug.to(memory_format=torch.channels_last).to(device=device)

    res2 = train.train_model(
        model=model_aug,
        device=device,
        dataset=dataset,
        epochs=40,
        batch_size=8,
        learning_rate=1e-5,
        img_scale=1.0,
        amp=True,
        # three-way split, by patient. split_seed must stay fixed across runs,
        # otherwise the "held-out" test patients change and stop being held out.
        val_percent=0.15,
        test_percent=0.15,
        split_seed=0,
        augment=True,           # rotation / elastic / contrast / noise on the training split only
        loss='dice_ce',         # identical to the baseline -- augmentation is the only variable
        select_on='macro_dice',
        lr_schedule='poly',     # identical to the baseline
        eval_test=False,        # test stays sealed
        run_name=run_name,
        phase=PHASE,             # picks the slice half + mask relabeling
        seed=seed,
    )
    aug_results[seed] = res2

    print(f'{run_name}: best val Dice (slice) {res2["best_val_dice"]:.4f}  (epoch {res2["best_epoch"]})')
    print(f'{run_name}: best val Dice (macro) {res2["best_val_macro_dice"]:.4f}   <- this is what selected best.pth')
    print(f'checkpoints           : {res2["checkpoint_dir"]}')

# Downstream cells were written against a single AUG_RUN / results2 -- point
# them at the same seed used for BASELINE_RUN above, so the paired comparison
# cell compares matching seeds.
AUG_RUN = aug_run_name(SEED)
results2 = aug_results[SEED]

## Data Augmentation: Rotation

In [ ]:
from utils.data_loading import AugmentedDataset
import matplotlib.pyplot as plt

# Get original sample
sample_original = train_set[0]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

# Original
axes[0].imshow(sample_original['image'][0], cmap='gray')
axes[0].imshow(sample_original['mask'], cmap='tab10', alpha=0.4)
axes[0].set_title("Original")
axes[0].axis("off")

# List of augmentations to visualize
augmentations = [
    ("Rotation",  {"rotate_prob": 1.0, "elastic_prob": 0.0, "contrast_prob": 0.0, "noise_prob": 0.0}),
    ("Elastic",   {"rotate_prob": 0.0, "elastic_prob": 1.0, "contrast_prob": 0.0, "noise_prob": 0.0}),
    ("Contrast",  {"rotate_prob": 0.0, "elastic_prob": 0.0, "contrast_prob": 1.0, "noise_prob": 0.0}),
    ("Noise",     {"rotate_prob": 0.0, "elastic_prob": 0.0, "contrast_prob": 0.0, "noise_prob": 1.0}),
]

# Generate one example for each augmentation
for ax, (title, probs) in zip(axes[1:], augmentations):
    aug_dataset = AugmentedDataset(train_set, **probs)
    sample = aug_dataset[0]

    ax.imshow(sample['image'][0], cmap='gray')
    ax.imshow(sample['mask'], cmap='tab10', alpha=0.4)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from utils.data_loading import AugmentedDataset
import matplotlib.pyplot as plt



plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
sample_original = train_set[0]
plt.imshow(sample_original['image'][0], cmap='gray')
plt.imshow(sample_original['mask'], cmap='tab10', alpha=0.4)

plt.subplot(1,2,2)
aug = AugmentedDataset(train_set, rotate_prob=0, elastic_prob=0, contrast_prob=1.0, noise_prob=0.0)  # force rotation every time, for inspection
sample = aug[0]
plt.imshow(sample['image'][0], cmap='gray')
plt.imshow(sample['mask'], cmap='tab10', alpha=0.4)

In [ ]:
# saving trained model
dir_checkpoint = train.dir_checkpoint
dir_checkpoint.mkdir(parents=True, exist_ok=True)

state_dict = model.state_dict()
state_dict['mask_values'] = dataset.mask_values
torch.save(state_dict, dir_checkpoint / 'model_final.pth')
print('Saved to', dir_checkpoint / 'model_final.pth')

In [ ]:
model = UNet(n_channels=1, n_classes=N_CLASSES, bilinear=False)
model = model.to(memory_format=torch.channels_last).to(device=device)

# best.pth is the epoch with the highest per-patient macro Dice (select_on='macro_dice'),
# not the highest slice-wise Dice. The slice-wise number scores a class that is
# absent from a slice as 1.0, so selecting on it rewards predicting nothing on
# empty slices.
checkpoint_path = run_dir_for(BASELINE_RUN) / 'best.pth'   # or AUG_RUN
state_dict = torch.load(checkpoint_path, map_location=device)
mask_values = state_dict.pop('mask_values')   # remove before loading, it's not a real weight
model.load_state_dict(state_dict)
model.eval()
print(f'Loaded {checkpoint_path}')

In [ ]:
from evaluate import evaluate

val_score = evaluate(model, val_loader, device, amp=False)
print(f'Final validation Dice score: {val_score:.4f}')

## Visualize predictions on a few validation slices

In [ ]:
import matplotlib
import IPython
#import matplotlib_inline

print("matplotlib:", matplotlib.__version__)
print("IPython:", IPython.__version__)
#print("matplotlib_inline:", matplotlib_inline.__version__)

In [ ]:
# USE VISLOADER for visulaisation (set drop_last to False), but use val_loader for loss
import matplotlib.pyplot as plt

model.eval()
vis_loader = DataLoader(val_set, shuffle=False, drop_last=False, batch_size=batch_size, num_workers=0, pin_memory=True)
batch = next(iter(vis_loader))

it = iter(vis_loader)
batch_num = 10

for _ in range(batch_num + 1):
    batch = next(it)

images, true_masks = batch['image'], batch['mask']

with torch.no_grad():
    images_dev = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
    preds = model(images_dev).argmax(dim=1).cpu()

n_show = min(4, images.shape[0])
fig, ax = plt.subplots(n_show, 3, figsize=(9, 3 * n_show), squeeze=False)
for i in range(n_show):
    ax[i, 0].imshow(images[i, 0], cmap='gray');   ax[i, 0].set_title('Image')
    ax[i, 1].imshow(true_masks[i], cmap='tab10', vmin=0, vmax=model.n_classes - 1); ax[i, 1].set_title('True mask')
    ax[i, 2].imshow(preds[i], cmap='tab10', vmin=0, vmax=model.n_classes - 1); ax[i, 2].set_title('Predicted mask')
    for a in ax[i]: a.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
for i in range(n_show):
    original_idx = val_idx[i]
    patient_id, slice_idx = dataset.index[original_idx]
    print(f'Row {i}: patient = {patient_id}, slice = {slice_idx}')

In [ ]:
# import matplotlib.pyplot as plt

# model.eval()
# batch = next(iter(val_loader))
# images, true_masks = batch['image'], batch['mask']

# with torch.no_grad():
#     images_dev = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)
#     preds = model(images_dev).argmax(dim=1).cpu()

# n_show = min(4, images.shape[0])
# fig, ax = plt.subplots(n_show, 3, figsize=(9, 3 * n_show))
# for i in range(n_show):
#     ax[i, 0].imshow(images[i, 0], cmap='gray');   ax[i, 0].set_title('Image')
#     ax[i, 1].imshow(true_masks[i], cmap='tab10'); ax[i, 1].set_title('True mask')
#     ax[i, 2].imshow(preds[i], cmap='tab10');      ax[i, 2].set_title('Predicted mask')
#     for a in ax[i]: a.axis('off')
# plt.tight_layout()
# plt.show()

### Per-class Dice breakdown

In [ ]:
# Per-class Dice + HD95, computed on whole 3D volumes, one value per patient.
# NOTE: the slice-wise dice_coeff scores a class that is ABSENT from a slice as
# 1.0, and 33.5% of (slice, class) pairs here are empty - so slice-wise per-class
# means are inflated. This scores whole volumes instead, where all four chambers
# are present in 100% of patients.
from utils import metrics

rows, summary = metrics.report(
    model, dataset, val_idx, device,
    n_classes=model.n_classes,
    split_name='val',
    class_names=train.PHASE_CLASS_NAMES[PHASE],     # {1: 'EAT'} for fat
    out_dir=None,                      # pass a Path to also write the CSVs
    amp=False,
    batch_size=batch_size,
)

# worst cases first - useful for spotting which patient/chamber is failing
print('\nper-patient detail (worst Dice first):')
for r in sorted(rows, key=lambda r: (r['dice'] != r['dice'], r['dice'])):
    hd = f"{r['hd95_mm']:.2f} mm" if r['hd95_mm'] == r['hd95_mm'] else 'n/a'
    print(f"  {r['patient_id']:<22}{r['class_name']:<4}dice={r['dice']:.4f}  hd95={hd}"
          + ('   <-- MISSED' if r['missed'] else ''))

## Reporting figures

In [ ]:
# Boxplots per structure - the standard segmentation results figure.
# Shows the outliers that a mean +/- SD hides (e.g. the one RA with HD95 ~44 mm).
#
# SPLIT = 'val': the test CSVs do not exist yet, because both runs above use
# eval_test=False. Switch to 'test' only after the gated cell at the bottom.
import csv
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

RUN = run_dir_for(BASELINE_RUN)
SPLIT = 'val'

rows = metrics.load_per_patient(RUN / f'{SPLIT}_metrics_per_patient.csv')
order = list(train.PHASE_CLASS_NAMES[PHASE].values())

# fixed generator so the jitter is identical every time this cell runs -- an
# unseeded one silently redraws the figure differently on each execution
jitter = np.random.default_rng(0)

panels = [('dice', 'Dice'), ('hd95_mm', 'HD95 (mm)'), ('assd_mm', 'ASSD (mm)')]
fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.5))
for ax, (key, label) in zip(axes, panels):
    if not rows or key not in rows[0]:
        ax.set_visible(False)          # metric absent from this CSV vintage
        continue
    data = [[r[key] for r in rows if r['class_name'] == c and r[key] == r[key]] for c in order]
    if not any(data):
        ax.set_visible(False)
        continue
    ax.boxplot(data, tick_labels=order, showmeans=True, widths=0.6)
    # overlay the individual patients - with n=7 the box spans only 3-4 points,
    # so the points are the data and the whiskers are close to decoration
    for i, vals in enumerate(data, start=1):
        ax.scatter(jitter.normal(i, 0.045, len(vals)), vals,
                   s=26, alpha=0.75, zorder=3, edgecolor='none')
    ax.set_ylabel(label)
    ax.set_title(f'{label} by chamber ({SPLIT}, n={max(len(v) for v in data)} patients)')
    ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Bland-Altman: agreement between predicted and reference chamber volume.
# Expected in cardiac papers - Dice tells you overlap, this tells you whether the
# model systematically over- or under-estimates the clinically relevant quantity.
fig, axes = plt.subplots(1, len(order), figsize=(4.75 * len(order), 4.2), sharey=False, squeeze=False)
axes = axes[0]

for ax, c in zip(axes, order):
    rs = [r for r in rows if r['class_name'] == c]
    gt = np.array([r['gt_voxels'] * r['voxel_ml'] for r in rs])
    pr = np.array([r['pred_voxels'] * r['voxel_ml'] for r in rs])
    mean_v, diff = (gt + pr) / 2, pr - gt
    bias, sd = diff.mean(), diff.std(ddof=1)

    ax.scatter(mean_v, diff, s=30)
    ax.axhline(bias, color='C1', label=f'bias {bias:+.1f} mL')
    ax.axhline(bias + 1.96 * sd, color='C3', ls='--', label=f'95% LoA +/-{1.96*sd:.1f}')
    ax.axhline(bias - 1.96 * sd, color='C3', ls='--')
    ax.axhline(0, color='grey', lw=0.8, zorder=0)
    ax.set_xlabel('mean of reference and predicted (mL)')
    ax.set_ylabel('predicted - reference (mL)')
    ax.set_title(f'{c}  (r={np.corrcoef(gt, pr)[0,1]:.3f})')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Seed-averaged per-patient rows, one side (baseline/aug) at a time.
#
# Averaging per (patient, class) FIRST matters once SEEDS has more than one
# entry: the same 7 val patients scored 3 times are not 21 independent
# observations -- pooling them would pseudoreplicate and understate the true
# p-value floor (2/2**7 = 0.0156 at n=7 PATIENTS, regardless of seed count).
# Average each patient's metric across its seeds first, then pair on patients
# below -- same approach as loss_ablation.ipynb's arm_rows()/SEED_MEAN_DIR.
def run_rows(run_name_fn):
    """Per-patient rows for one run type (baseline/aug), averaged over
    whatever seeds have finished val scoring so far."""
    by_key, found = {}, []
    for seed in SEEDS:
        path = run_dir_for(run_name_fn(seed)) / 'val_metrics_per_patient.csv'
        if not path.exists():
            continue
        found.append(seed)
        for r in metrics.load_per_patient(path):
            by_key.setdefault((r['patient_id'], r['class_name']), []).append(r)

    out = []
    for (pid, cname), rs in sorted(by_key.items()):
        row = {'patient_id': pid, 'class_name': cname, 'cls': rs[0]['cls']}
        for m in metrics.DEFAULT_METRICS + ('voxel_ml',):
            if m not in rs[0]:
                continue
            vals = [r[m] for r in rs if np.isfinite(r[m])]
            row[m] = float(np.mean(vals)) if vals else float('nan')
        for m in ('gt_voxels', 'pred_voxels'):
            row[m] = int(np.mean([r[m] for r in rs]))
        row['missed'] = any(r['missed'] for r in rs)
        row['n_seeds'] = len(rs)
        out.append(row)
    return out, set(found)


SEED_MEAN_DIR = train.dir_checkpoint / 'fat_baseline_vs_aug_seedmean'
SEED_MEAN_DIR.mkdir(parents=True, exist_ok=True)

baseline_rows, baseline_seeds = run_rows(baseline_run_name)
aug_rows, aug_seeds = run_rows(aug_run_name)

# Unbalanced seeds are a silent confound: if one side carries 3 seeds and the
# other carries 1, the gap between them is part augmentation and part which
# init each happened to draw. Only compare on the seeds both sides have.
if aug_seeds and baseline_seeds != aug_seeds:
    shared = sorted(baseline_seeds & aug_seeds)
    print(f'WARNING: baseline seeds {sorted(baseline_seeds)} != aug seeds '
          f'{sorted(aug_seeds)}. Run the missing seed(s), or set '
          f'SEEDS = {shared} and re-run this cell before comparing.')

baseline_csv = SEED_MEAN_DIR / 'baseline_val_metrics_per_patient.csv'
aug_csv = SEED_MEAN_DIR / 'aug_val_metrics_per_patient.csv'
metrics.write_csv(baseline_rows, baseline_csv)   # compare_runs reads CSVs, not objects
metrics.write_csv(aug_rows, aug_csv)

print(f'baseline: seeds {sorted(baseline_seeds)}, {len(baseline_rows)} (patient, class) rows')
print(f'aug     : seeds {sorted(aug_seeds)}, {len(aug_rows)} (patient, class) rows')
print('seed-averaged CSVs written to', SEED_MEAN_DIR)

In [ ]:
# Paired comparison of baseline vs augmentation on the SAME validation patients,
# each side averaged over its finished seeds FIRST (cell above) -- so this
# isn't confounded with which init one particular seed happened to draw.
#
# Validation, not test: choosing between augment=False and augment=True is model
# selection, so it happens on val. The test set is scored once, in the gated cell
# below, using whichever of the two you pick.
#
# Sample-size warning, applied automatically by compare_runs: the exact
# two-sided Wilcoxon floor at n=7 patients is 2/2**7 = 0.0156. The fat model has
# only ONE foreground class (EAT), so the "per-chamber" family here is just
# 1 class x 3 metrics (dice, hd95_mm, assd_mm) = 3 tests -- Holm-corrected floor
# 0.0156 * 3 = 0.047, which unlike the 4-chamber water model CAN survive at
# p < 0.05. The macro row is identical to the EAT row when there is only one
# foreground class, and is left uncorrected as the single primary endpoint.
if aug_csv.exists() and aug_rows:
    comparison = metrics.compare_runs(baseline_csv, aug_csv, label_a='base', label_b='aug')
    print(metrics.format_comparison(comparison, 'base', 'aug'))
else:
    print('No augmented runs finished yet.')
    print('Run the augment=True cell above, keeping split_seed=0 so both are')
    print('scored on the same validation patients.')

## Custom multi-run summary

Combine any set of runs you choose into one `mean +/- SD` table, similar to
`ablation_val_summary.csv` in `loss_ablation.ipynb` but without that notebook's
arm/seed naming convention — just list the runs you want.

In [ ]:
# Point RUNS at whichever runs you want in the table. A value is either a run
# directory under train.dir_checkpoint (its '{SPLIT_NAME}_metrics_per_patient.csv'
# is used) or a full/relative path straight to a per-patient CSV.
from utils import metrics

RUNS = {
    'baseline':  'baseline_dice_ce2',
    'augmented': 'data_aug_dice_ce2',
    # 'custom':  'checkpoints/some_other_run/val_metrics_per_patient.csv',
}
SPLIT_NAME = 'val'   # only used for entries given as a run directory, not a CSV path

import json

def _resolve_csv(value):
    p = Path(value)
    if p.suffix == '.csv':
        return p
    run_dir = p if p.is_absolute() else train.dir_checkpoint / p
    return run_dir / f'{SPLIT_NAME}_metrics_per_patient.csv'

combined = []
split_seeds = {}
for label, value in RUNS.items():
    csv_path = _resolve_csv(value)
    if not csv_path.exists():
        print(f'{label}: {csv_path} not found, skipped')
        continue

    cfg_path = csv_path.parent / 'run_config.json'
    if cfg_path.exists():
        split_seeds[label] = json.loads(cfg_path.read_text()).get('split_seed')

    rows = metrics.load_per_patient(csv_path)
    for r in metrics.summarise(rows):
        out = {'run': label, 'class_name': r['class_name'], 'cls': r['cls'],
               'n_patients': r['n_patients'], 'n_missed': r['n_missed']}
        for m in metrics.DEFAULT_METRICS:
            if f'{m}_mean' not in r:
                continue
            for stat in ('mean', 'sd', 'median', 'n'):
                out[f'{m}_{stat}'] = r[f'{m}_{stat}']
        combined.append(out)

# runs scored on different patients aren't comparable -- warn rather than
# silently tabling them side by side
if len({s for s in split_seeds.values() if s is not None}) > 1:
    print(f'WARNING: runs used different split_seeds: {split_seeds}')

out_path = train.dir_checkpoint / 'custom_val_summary.csv'
metrics.write_csv(combined, out_path)
print(f'wrote {len(combined)} rows to {out_path}\n')

print(f'{"run":<20}{"class":<20}{"Dice":>17}{"Prec":>17}{"Rec":>17}{"HD95 mm":>17}{"n":>4}{"missed":>8}')
for r in combined:
    cells = ''
    for m, dp in [('dice', 4), ('precision', 4), ('recall', 4), ('hd95_mm', 2)]:
        cells += (f'{r[f"{m}_mean"]:>10.{dp}f}+/-{r[f"{m}_sd"]:<4.2f}'
                  if f'{m}_mean' in r else f'{"-":>17}')
    print(f'{r["run"]:<20}{r["class_name"]:<20}{cells}{r["n_patients"]:>4}{r["n_missed"]:>8}')

## Final: score the held-out test set, once

Everything above runs on validation. Run this **after** you have chosen between
baseline and augmentation (and, if you are also running `loss_ablation.ipynb`,
after the loss is settled), and run it once.

It reloads `best.pth` — weights selected on validation that have never seen a
test patient — and scores the 7 test patients, writing
`test_metrics_per_patient.csv` and `test_metrics_summary.csv` into the run
directory. Report the result; do not go back and change a setting because of it.

**Note on the pre-existing runs.** `checkpoints/baseline_3split/` and
`checkpoints/data_aug_3split/` already contain `test_metrics_*.csv` — those runs
scored test automatically under the old default (`eval_test=True`). The test set
has therefore already been looked at twice. That cannot be undone; treat the
number this cell produces as a held-out estimate with that caveat attached, and
keep test sealed from here on.

In [ ]:
WINNING_RUN = BASELINE_RUN     # <- set to AUG_RUN if augmentation won on validation
RUN_FINAL_TEST = False         # <- flip to True deliberately, once

if RUN_FINAL_TEST:
    run_dir = run_dir_for(WINNING_RUN)

    final_model = UNet(n_channels=1, n_classes=N_CLASSES, bilinear=False)
    final_model = final_model.to(memory_format=torch.channels_last).to(device=device)
    sd = torch.load(run_dir / 'best.pth', map_location=device)
    sd.pop('mask_values', None)          # injected by train.py, not a real weight
    final_model.load_state_dict(sd)
    final_model.eval()

    rows_test, summary_test = metrics.report(
        final_model, dataset, test_idx, device,
        n_classes=N_CLASSES,
        out_dir=run_dir, split_name='test',
        class_names=train.PHASE_CLASS_NAMES[PHASE],
        amp=True, batch_size=batch_size,
    )
    print(f'FINAL -- {WINNING_RUN} on {len(set(dataset.index[i][0] for i in test_idx))} '
          f'held-out test patients:\n')
    print(metrics.format_summary(summary_test))
    print('\nCSVs written to', run_dir)
else:
    print('Test evaluation is gated. Set RUN_FINAL_TEST = True once baseline vs '
          'augmentation is decided.')

In [ ]:
# sanity check: the three sets must be disjoint and cover every slice
print('train slices:', len(train_idx))
print('val slices  :', len(val_idx))
print('test slices :', len(test_idx))
print('total       :', len(dataset.index))

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0
assert len(train_idx) + len(val_idx) + len(test_idx) == len(dataset.index)

pat = lambda idxs: {dataset.index[i][0] for i in idxs}
assert not (pat(train_idx) & pat(val_idx)) and not (pat(train_idx) & pat(test_idx))
print('OK: disjoint at slice AND patient level, every slice covered once')

In [ ]:
import pydicom
ds = pydicom.dcmread(train.dir_img / 'CADRE_1872.dcm')
print(ds.pixel_array.ndim, ds.pixel_array.shape)

In [ ]:
# (duplicate of the split cell above - kept so the cells below still run
# if you jump straight here)
dataset = VolumeMRIDataset(train.dir_img, train.dir_mask, scale=img_scale, phase=PHASE)

train_idx, val_idx, test_idx = train.split_patients(
    dataset, val_percent=val_percent, test_percent=test_percent, seed=split_seed
)
train_set = Subset(dataset, train_idx)
val_set   = Subset(dataset, val_idx)
test_set  = Subset(dataset, test_idx)

print('val patients :', sorted({dataset.index[i][0] for i in val_idx}))
print('test patients:', sorted({dataset.index[i][0] for i in test_idx}))
print('val_set size:', len(val_set))

In [ ]:
import pydicom

ds = pydicom.dcmread(train.dir_img / 'CADRE_1598.dcm')
print('pixel_array shape:', ds.pixel_array.shape)
print('SamplesPerPixel:', getattr(ds, 'SamplesPerPixel', 'not set'))
print('PhotometricInterpretation:', getattr(ds, 'PhotometricInterpretation', 'not set'))

In [ ]:
import numpy as np
frame = ds.pixel_array[0]  # shape (432, 432, 3)
print(np.allclose(frame[..., 0], frame[..., 1]), np.allclose(frame[..., 1], frame[..., 2]))